In [ ]:
import os
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
from torch.optim.lr_scheduler import LambdaLR
import math
import multiprocessing

# ======================
# 设备配置
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# 配置参数
# ======================
class Config:
    input_seq_len = 18       # delta_x, delta_y, h(16)
    output_seq_len = 16      # 仅输出 h(16)
    d_model = 256
    num_layers = 6
    dim_feedforward = 1024
    dropout = 0.1
    batch_size = 256
    num_epochs = 100
    learning_rate = 3e-4
    warmup_steps = 4000
    grad_clip = 1
    early_stop_patience = 15
    model_save_path = "models/m-s2s-b2d-atten.pth"
    combined_loss_path = "loss/loss-s2s-b2d-atten.txt"
    test_data_path = "data/data-3-tes.json"
    random_sample_size = 100000

config = Config()

# ======================
# 确保目录存在
# ======================
os.makedirs(os.path.dirname(config.model_save_path), exist_ok=True)
os.makedirs(os.path.dirname(config.combined_loss_path), exist_ok=True)
os.makedirs(os.path.dirname(config.test_data_path), exist_ok=True)

# ======================
# 数据集定义
# ======================
class PathDataset(Dataset):
    def __init__(self, file_path, sample_size=None):
        with open(file_path, 'r') as f:
            full_data = json.load(f)

        if sample_size and sample_size < len(full_data):
            self.data = random.sample(full_data, sample_size)
        else:
            self.data = full_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # ✅ 输入：delta_x, delta_y, h(16)
        input_vec = torch.FloatTensor([
            item['input']['delta_x'],
            item['input']['delta_y'],
            *item['input']['h']
        ])  # [18]

        # ✅ 输出：仅 h(16)
        output_vec = torch.FloatTensor(item['output']['h'])  # [16]

        return input_vec, output_vec

    def save_subset_to_json(self, indices, output_path):
        subset_data = [self.data[i] for i in indices]
        with open(output_path, 'w') as f:
            json.dump(subset_data, f, indent=2)


# ======================
# 模型定义
# ======================
class DirectMappingTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        # 输入为 18 个 token，每个 token 映射到 d_model
        self.input_proj = nn.Linear(1, config.d_model)

        # Transformer 编码器层
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=8,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers)

        # 全局平均池化，将输入18个token的输出压缩到16个token长度
        self.global_pool = nn.AdaptiveAvgPool1d(config.output_seq_len)

        # 输出层：每个token生成一个标量（对应h的一个维度）
        self.output_proj = nn.Sequential(
            nn.Linear(config.d_model, config.d_model),
            nn.ReLU(),
            nn.Linear(config.d_model, 1)
        )

    def forward(self, src):
        # src: [B, 18]
        src = src.unsqueeze(-1)               # [B, 18, 1]
        src = self.input_proj(src)            # [B, 18, d_model]
        encoded = self.encoder(src)           # [B, 18, d_model]
        encoded = encoded.transpose(1, 2)     # [B, d_model, 18]
        pooled = self.global_pool(encoded)    # [B, d_model, 16]
        pooled = pooled.transpose(1, 2)       # [B, 16, d_model]
        out = self.output_proj(pooled).squeeze(-1)  # [B, 16]
        return out


# ======================
# 学习率调度
# ======================
def get_lr_scheduler(optimizer, train_loader_len):
    def lr_lambda(step):
        if step < config.warmup_steps:
            return step / float(max(1, config.warmup_steps))
        progress = (step - config.warmup_steps) / float(max(1, config.num_epochs * train_loader_len - config.warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


# ======================
# 精度计算函数
# ======================
def compute_accuracy(output, target, threshold=0.01):
    abs_diff = torch.abs(output - target)
    correct = torch.sum(abs_diff < threshold, dim=1)
    return torch.mean(correct.float() / output.size(1)).item()


# ======================
# 训练与验证
# ======================
def train_model(model, dataloader, optimizer, criterion, scheduler=None):
    model.train()
    total_loss, total_acc = 0.0, 0.0
    for src, tgt in tqdm(dataloader, desc="Training", leave=True):
        src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(src)
        loss = criterion(outputs, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        optimizer.step()
        if scheduler: scheduler.step()
        total_loss += loss.item()
        total_acc += compute_accuracy(outputs, tgt)
    return total_loss / len(dataloader), total_acc / len(dataloader)


def validate_model(model, dataloader, criterion):
    model.eval()
    total_loss, total_acc = 0.0, 0.0
    with torch.no_grad():
        for src, tgt in tqdm(dataloader, desc="Validating", leave=True):
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            outputs = model(src)
            loss = criterion(outputs, tgt)
            total_loss += loss.item()
            total_acc += compute_accuracy(outputs, tgt)
    return total_loss / len(dataloader), total_acc / len(dataloader)


# ======================
# 主函数
# ======================
if __name__ == "__main__":
    multiprocessing.freeze_support()
    torch.multiprocessing.set_sharing_strategy('file_system')

    print("✅ 加载数据...")
    dataset = PathDataset("data/data-2-s2s.json", config.random_sample_size)
    print(f"样本数: {len(dataset)}")

    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_ds, val_ds, test_ds = random_split(
        dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    dataset.save_subset_to_json(test_ds.indices, config.test_data_path)

    print("✅ 启动 DataLoader (num_workers=4, pin_memory=True)...")
    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                              num_workers=6, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False,
                            num_workers=6, pin_memory=True)

    print("✅ 初始化模型...")
    model = DirectMappingTransformer().to(device)
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    scheduler = get_lr_scheduler(optimizer, len(train_loader))
    criterion = nn.MSELoss()

    best_val_loss, early_stop_counter = float('inf'), 0
    with open(config.combined_loss_path, 'w') as f:
        f.write("Epoch,TrainLoss,TrainAcc,ValLoss,ValAcc\n")

    print("🚀 开始训练...")
    for epoch in range(config.num_epochs):
        train_loss, train_acc = train_model(model, train_loader, optimizer, criterion, scheduler)
        val_loss, val_acc = validate_model(model, val_loader, criterion)

        with open(config.combined_loss_path, 'a') as f:
            f.write(f"{epoch+1},{train_loss:.6f},{train_acc:.4f},{val_loss:.6f},{val_acc:.4f}\n")

        print(f"\nEpoch {epoch+1}/{config.num_epochs} | "
              f"Train Loss: {train_loss:.6f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.6f} Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stop_counter = 0
            torch.save(model.state_dict(), config.model_save_path)
            print(f"💾 模型保存至 {config.model_save_path}")
        else:
            early_stop_counter += 1
            if early_stop_counter >= config.early_stop_patience:
                print(f"⏹ Early stopping at epoch {epoch+1}")
                break

    print(f"✅ 训练完成. 最佳验证损失: {best_val_loss:.6f}")


✅ 加载数据...
样本数: 100000
✅ 启动 DataLoader (num_workers=4, pin_memory=True)...
✅ 初始化模型...
🚀 开始训练...


Validating: 100%|██████████| 40/40 [00:00<00:00, 101.06it/s]



Epoch 1/100 | Train Loss: 0.898903 Acc: 0.0288 | Val Loss: 0.734543 Acc: 0.0367
💾 模型保存至 models/m-s2s-b2d-atten.pth


Validating: 100%|██████████| 40/40 [00:00<00:00, 107.28it/s]



Epoch 2/100 | Train Loss: 0.657848 Acc: 0.0362 | Val Loss: 0.600842 Acc: 0.0462
💾 模型保存至 models/m-s2s-b2d-atten.pth


...


Epoch 98/100 | Train Loss: 0.121983 Acc: 0.0924 | Val Loss: 0.092354 Acc: 0.1191


Validating: 100%|██████████| 40/40 [00:00<00:00, 98.72it/s]



Epoch 99/100 | Train Loss: 0.122372 Acc: 0.0925 | Val Loss: 0.092668 Acc: 0.1194


Validating: 100%|██████████| 40/40 [00:00<00:00, 85.56it/s] 


Epoch 100/100 | Train Loss: 0.121922 Acc: 0.0924 | Val Loss: 0.092484 Acc: 0.1194
✅ 训练完成. 最佳验证损失: 0.091983


## 添加位置编码

In [1]:
import os
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
from torch.optim.lr_scheduler import LambdaLR
import math
import multiprocessing

# ======================
# 设备配置
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# 配置参数
# ======================
class Config:
    input_seq_len = 18       # delta_x, delta_y, h(16)
    output_seq_len = 16      # 仅输出 h(16)
    d_model = 512
    num_layers = 2
    dim_feedforward = 1024
    dropout = 0.1
    batch_size = 256
    num_epochs = 150
    learning_rate = 3e-4
    warmup_steps = 4000
    grad_clip = 1
    early_stop_patience = 50
    model_save_path = "models/m-s2s-b2d-atten-pos.pth"
    combined_loss_path = "loss/loss-s2s-b2d-atten-pos.txt"
    test_data_path = "data/data-3-tes.json"
    random_sample_size = 100000

config = Config()

# ======================
# 确保目录存在
# ======================
os.makedirs(os.path.dirname(config.model_save_path), exist_ok=True)
os.makedirs(os.path.dirname(config.combined_loss_path), exist_ok=True)
os.makedirs(os.path.dirname(config.test_data_path), exist_ok=True)

# ======================
# 数据集定义
# ======================
class PathDataset(Dataset):
    def __init__(self, file_path, sample_size=None):
        with open(file_path, 'r') as f:
            full_data = json.load(f)

        if sample_size and sample_size < len(full_data):
            self.data = random.sample(full_data, sample_size)
        else:
            self.data = full_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # ✅ 输入：delta_x, delta_y, h(16)
        input_vec = torch.FloatTensor([
            item['input']['delta_x'],
            item['input']['delta_y'],
            *item['input']['h']
        ])  # [18]

        # ✅ 输出：仅 h(16)
        output_vec = torch.FloatTensor(item['output']['h'])  # [16]

        return input_vec, output_vec

    def save_subset_to_json(self, indices, output_path):
        subset_data = [self.data[i] for i in indices]
        with open(output_path, 'w') as f:
            json.dump(subset_data, f, indent=2)


# ======================
# 模型定义（添加位置编码）
# ======================
class DirectMappingTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        # 输入为 18 个 token，每个 token 映射到 d_model
        self.input_proj = nn.Linear(1, config.d_model)

        # 位置编码 - 可学习的参数
        self.pos_encoding = nn.Parameter(torch.zeros(1, config.input_seq_len, config.d_model))

        # Transformer 编码器层
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=8,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers)

        # 全局平均池化，将输入18个token的输出压缩到16个token长度
        self.global_pool = nn.AdaptiveAvgPool1d(config.output_seq_len)

        # 输出层：每个token生成一个标量（对应h的一个维度）
        self.output_proj = nn.Sequential(
            nn.Linear(config.d_model, config.d_model),
            nn.ReLU(),
            nn.Linear(config.d_model, 1)
        )

    def forward(self, src):
        # src: [B, 18]
        src = src.unsqueeze(-1)               # [B, 18, 1]
        src = self.input_proj(src)            # [B, 18, d_model]
        
        # 添加位置编码
        src = src + self.pos_encoding         # [B, 18, d_model]
        
        encoded = self.encoder(src)           # [B, 18, d_model]
        encoded = encoded.transpose(1, 2)     # [B, d_model, 18]
        pooled = self.global_pool(encoded)    # [B, d_model, 16]
        pooled = pooled.transpose(1, 2)       # [B, 16, d_model]
        out = self.output_proj(pooled).squeeze(-1)  # [B, 16]
        return out


# ======================
# 学习率调度
# ======================
def get_lr_scheduler(optimizer, train_loader_len):
    def lr_lambda(step):
        if step < config.warmup_steps:
            return step / float(max(1, config.warmup_steps))
        progress = (step - config.warmup_steps) / float(max(1, config.num_epochs * train_loader_len - config.warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


# ======================
# 精度计算函数
# ======================
def compute_accuracy(output, target, threshold=0.01):
    abs_diff = torch.abs(output - target)
    correct = torch.sum(abs_diff < threshold, dim=1)
    return torch.mean(correct.float() / output.size(1)).item()


# ======================
# 训练与验证
# ======================
def train_model(model, dataloader, optimizer, criterion, scheduler=None):
    model.train()
    total_loss, total_acc = 0.0, 0.0
    for src, tgt in tqdm(dataloader, desc="Training", leave=True):
        src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(src)
        loss = criterion(outputs, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        optimizer.step()
        if scheduler: scheduler.step()
        total_loss += loss.item()
        total_acc += compute_accuracy(outputs, tgt)
    return total_loss / len(dataloader), total_acc / len(dataloader)


def validate_model(model, dataloader, criterion):
    model.eval()
    total_loss, total_acc = 0.0, 0.0
    with torch.no_grad():
        for src, tgt in tqdm(dataloader, desc="Validating", leave=True):
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            outputs = model(src)
            loss = criterion(outputs, tgt)
            total_loss += loss.item()
            total_acc += compute_accuracy(outputs, tgt)
    return total_loss / len(dataloader), total_acc / len(dataloader)


# ======================
# 主函数
# ======================
if __name__ == "__main__":
    multiprocessing.freeze_support()
    torch.multiprocessing.set_sharing_strategy('file_system')

    print("✅ 加载数据...")
    dataset = PathDataset("data/data-2-s2s.json", config.random_sample_size)
    print(f"样本数: {len(dataset)}")

    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_ds, val_ds, test_ds = random_split(
        dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    dataset.save_subset_to_json(test_ds.indices, config.test_data_path)

    print("✅ 启动 DataLoader (num_workers=4, pin_memory=True)...")
    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                              num_workers=6, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False,
                            num_workers=6, pin_memory=True)

    print("✅ 初始化模型...")
    model = DirectMappingTransformer().to(device)
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    scheduler = get_lr_scheduler(optimizer, len(train_loader))
    criterion = nn.MSELoss()

    best_val_loss, early_stop_counter = float('inf'), 0
    with open(config.combined_loss_path, 'w') as f:
        f.write("Epoch,TrainLoss,TrainAcc,ValLoss,ValAcc\n")

    print("🚀 开始训练...")
    for epoch in range(config.num_epochs):
        train_loss, train_acc = train_model(model, train_loader, optimizer, criterion, scheduler)
        val_loss, val_acc = validate_model(model, val_loader, criterion)

        with open(config.combined_loss_path, 'a') as f:
            f.write(f"{epoch+1},{train_loss:.6f},{train_acc:.4f},{val_loss:.6f},{val_acc:.4f}\n")

        print(f"\nEpoch {epoch+1}/{config.num_epochs} | "
              f"Train Loss: {train_loss:.6f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.6f} Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stop_counter = 0
            torch.save(model.state_dict(), config.model_save_path)
            print(f"💾 模型保存至 {config.model_save_path}")
        else:
            early_stop_counter += 1
            if early_stop_counter >= config.early_stop_patience:
                print(f"⏹ Early stopping at epoch {epoch+1}")
                break

    print(f"✅ 训练完成. 最佳验证损失: {best_val_loss:.6f}")

✅ 加载数据...
样本数: 100000
✅ 启动 DataLoader (num_workers=4, pin_memory=True)...
✅ 初始化模型...
🚀 开始训练...


Validating: 100%|██████████| 40/40 [00:00<00:00, 69.39it/s]



Epoch 1/150 | Train Loss: 0.821715 Acc: 0.0285 | Val Loss: 0.238148 Acc: 0.0306
💾 模型保存至 models/m-s2s-b2d-atten-pos.pth


Validating: 100%|██████████| 40/40 [00:00<00:00, 67.01it/s]



Epoch 2/150 | Train Loss: 0.216707 Acc: 0.0467 | Val Loss: 0.096956 Acc: 0.1032
💾 模型保存至 models/m-s2s-b2d-atten-pos.pth


...


Epoch 148/150 | Train Loss: 0.000111 Acc: 0.8207 | Val Loss: 0.000770 Acc: 0.7634


Validating: 100%|██████████| 40/40 [00:00<00:00, 66.79it/s]



Epoch 149/150 | Train Loss: 0.000111 Acc: 0.8210 | Val Loss: 0.000774 Acc: 0.7650


Validating: 100%|██████████| 40/40 [00:00<00:00, 68.28it/s]


Epoch 150/150 | Train Loss: 0.000111 Acc: 0.8202 | Val Loss: 0.000775 Acc: 0.7644
✅ 训练完成. 最佳验证损失: 0.000638
